In [ ]:
!pip install faster-whisper

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path

In [ ]:
def format_timestamp(seconds: float) -> str:
    """
    Convert seconds to SRT timestamp format: HH:MM:SS,mmm
    """
    if seconds is None:
        seconds = 0.0

    milliseconds = int(round(seconds * 1000))
    hours = milliseconds // 3_600_000
    milliseconds %= 3_600_000
    minutes = milliseconds // 60_000
    milliseconds %= 60_000
    secs = milliseconds // 1000
    milliseconds %= 1000

    return f"{hours:02}:{minutes:02}:{secs:02},{milliseconds:03}"


def write_srt(segments, output_path: str) -> None:
    """
    Write transcription segments to an .srt file
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for i, segment in enumerate(segments, start=1):
            start = format_timestamp(segment.start)
            end = format_timestamp(segment.end)
            text = segment.text.strip()

            f.write(f"{i}\n")
            f.write(f"{start} --> {end}\n")
            f.write(f"{text}\n\n")

In [ ]:
def generate_subtitles(
    video_path: str,
    model_size: str = "small",
    language: str = None,
    output_srt: str = None,
    device: str = "cpu",
    compute_type: str = "int8",
):
    """
    Transcribe a video file and save subtitles as .srt
    """

    video_file = Path(video_path)

    if not video_file.exists():
        raise FileNotFoundError(f"File not found: {video_path}")

    if output_srt is None:
        output_srt = str(video_file.with_suffix(".srt"))

    model = WhisperModel(model_size, device=device, compute_type=compute_type)

    segments, info = model.transcribe(
        str(video_file),
        language=language,
        beam_size=5
    )

    segments = list(segments)
    write_srt(segments, output_srt)

    print(f"detected language: {info.language}")
    print(f"language probability: {info.language_probability:.2f}")
    print(f"saved subtitles to: {output_srt}")

    return output_srt

In [ ]:
video_path = "My_Video.mp4"

In [ ]:
output_file = generate_subtitles(
    video_path=video_path,
    model_size="medium",   # try "medium" for better accuracy
    language=None,        # or "en" to force english
    device="cpu",
    compute_type="int8"
)

output_file